# Using TigerGraph GraphRAG for Document Question Answering

This notebook demostrates how to use TigerGraph GraphRAG, an AI assistant for your TigerGraph databases. TigerGraph GraphRAG enables you to ask questions in natural language about your document data stored in TigerGraph and get answers in a human-readable format. GraphRAG is a graph-based retrieval-augmented generation approach that is used to answer questions about the document data stored in TigerGraph. TigerGraph GraphRAG is built to help users get started with GraphRAG and to provide a seamless experience for users to interact with their document data within TigerGraph.

In [7]:

import os
from pyTigerGraph import TigerGraphConnection
from dotenv import load_dotenv

load_dotenv()
# We first create a connection to the database
tg_host = "http://localhost" 
username = os.getenv("USERNAME", "tigergraph")
password = os.getenv("PASS", "tigergraph")
conn = TigerGraphConnection(
    host=tg_host,
    username=username,
    password=password,
    gsPort="14240",
    restppPort="14240",
    graphname = "corpus5"
)

# And then add GraphRAG's address to the connection. This address
# is the host's address where the GraphRAG container is running.
graphrag_host = "http://localhost"
conn.ai.configureGraphRAGHost(f"{graphrag_host}:8000")

## Create a Graph and Ingest Data

We provide utilities to setup your TigerGraph database with a schema and load your desired documents. In this example, we are utilizing the TigerGraph documentation as our dataset. The documents need to be processed into a JSONL file of the following format in order to load into TigerGraph:

```json
{"doc_id": "id_for_document_here", "doc_type": "content type", "content": "Text content of the document"}
```
The `doc_type` field determines the chunker to be used for the content chunking strategy. Currently it supports `semantic`, `markdown`, `html`, `regex`, `character`, `recursive`, `single` and `image`. Leave it blank will use `semantic` chunker by default.

The following code block will create a graph called `TigerGraphRAG` and load the documents into the graph. The schema that is created looks like this:

![graphrag_schema](../img/GraphRAGSchema.png)

Create Knowledge Graph

In [9]:
conn.gsql(f"""CREATE GRAPH {conn.graphname}()""")

TigerGraphException: ('User authentication failed', None)

Get connection token if authentication is enabled

In [ ]:
# We need to authenticate the connection if restpp authentication is enabled
conn.getToken()

TigerGraphException: ('User authentication failed', None)

Create SuportAI schema and install related queries

In [ ]:
conn.ai.initializeGraphRAG()

NameError: name 'conn' is not defined

Create DocumentIngest for local file

In [ ]:
res = conn.ai.createDocumentIngest(
    data_source="local",
    data_source_config={"data_path": "../data/tg_tutorials.jsonl"},
    file_format="json",
)

Run DocumentIngest to load documents to graph

In [ ]:
conn.ai.runDocumentIngest(res["load_job_id"], res["data_source_id"], res["data_path"])

Alternatively, create and run DocumentIngest for data files on Cloud storage

In [ ]:
access = ""
sec = ""
res = conn.ai.createDocumentIngest(
    data_source="s3",
    data_source_config={"aws_access_key": access, "aws_secret_key": sec},
    loader_config={"doc_id_field": "url", "content_field": "content", "doc_type": ""},
    file_format="json",
)

In [ ]:
conn.ai.runDocumentIngest(res["load_job_id"], res["data_source_id"], "s3://tg-documentation/pytg_current/pytg_current.jsonl")

When using AWS Bedrock BDA for multimodal data ingestion, run the following step only:

In [ ]:
access = ""
sec = ""
region_name = "us-west-2"
input_bucket = "bedrock-graphrag-data"
output_bucket = "bedrock-graphrag-output"
res = conn.ai.createDocumentIngest(
    data_source="bda",
    data_source_config={"aws_access_key": access, "aws_secret_key": sec, "region_name": region_name, "input_bucket": input_bucket, "output_bucket": output_bucket},
)

In [ ]:
conn.ai.runDocumentIngest(res["load_job_id"], res["data_source_id"])

## Build Knowledge Graph from the documents loaded

In [ ]:
conn.ai.forceConsistencyUpdate("graphrag")

## Comparing Document Search Methods
TigerGraph GraphRAG provides multiple methods to search documents in the graph. The methods are:

- **Similarity Search**: This is the baseline, traditional vector search approach. It compares the query embedding against the document chunk embeddings, and returns the closest top-k matches. While efficient, this method is limited to semantic similarity and does not capture additional context or structural connections.

- **Contextual Search**: This method enhances vector search by incorporating graph traversal via sibling relationships (e.g., IS_AFTER) between document chunks. Starting from the top-k most relevant chunk, it expands to include surrounding chunks, ensuring that context spanning across multiple paragraphs or sections is retrieved. This provides a more complete answer when key information is distributed across adjacent text.

- **Hybrid Search**: Hybrid Search: Hybrid search combines vector search with graph traversal via common entities and relationships among document chunks. After identifying the top-k most relevant documents chunks and entities, it expands through the graph by leveraging contextual, entity, and containment relationships. This integration of semantic similarity with structural connections enables more comprehensive and precise retrieval.

- **Community Search**: Community search extends retrieval capabilities by combining graph traversal with layered community detection. It begins with the top-k most similar communities, then traverses relevant relationships within these communities to surface both related communities and their associated document chunks. Unlike purely semantic methods, this approach leverages hierarchical community patterns to deliver richer, more coherent context. Community-based search is especially effective for broader queries, where balancing scope and detail is critical.

In [ ]:
query = "According to the provided corpus, which athletics event at the 2008 Summer Olympics had the highest number of competitors?"

### Hybrid Search in TigerGraph

In [ ]:
conn.ai.searchDocuments(query,
                        method="hybrid",
                        method_parameters = {"indices": ["DocumentChunk", "Entity"],
                                             "top_k": 5,
                                             "num_hops": 2,
                                             "num_seen_min": 3,
                                             "verbose": False})

HTTPError: 500 Server Error: Internal Server Error for url: http://localhost:8000/corpus5/graphrag/search

### Document Chunk Similarity Search

In [ ]:
conn.ai.searchDocuments(query,
                        method="similarity",
                        method_parameters={"index": "DocumentChunk",
                                           "top_k": 5,
                                           "withHyDE": False,
                                           "verbose": False})

### Sibling Document Chunk Similarity Search

In [ ]:
conn.ai.searchDocuments(query,
                        method="contextual",
                        method_parameters={"index": "DocumentChunk",
                                           "top_k": 5,
                                           "lookahead": 3,
                                           "lookback": 3,
                                           "withHyDE": False,
                                           "verbose": False})

### GraphRAG Document Chunk Community Search

In [ ]:
conn.ai.searchDocuments(query,
                        method="community",
                        method_parameters={"community_level": 2, "top_k": 3, "verbose": True})

## Comparing LLM Generated Responses

TigerGraph GraphRAG provides a way to generate the response to the user's query using a LLM, based on the search results from the methods above. You can compare the responses generated by the LLM for each of the search methods to see which one is the most relevant to the user's query. In this example, we can see that the Hybrid Search method generates the most relevant response to the user's query. While none of the responses were wrong, the Hybrid Search method generated the most relevant response to the user's query, by suggesting to use the `getVertexCount()` function to get the number of vertices in the graph.

In [ ]:
resp = conn.ai.answerQuestion(query,
                        method="community",
                        method_parameters={"community_level": 2, "top_k": 3, "verbose": True})

In [ ]:
print(resp["response"])

Check verbose info for more details if needed

In [ ]:
import json
print(json.dumps(resp["verbose"], indent=4))

### Answer question using Hybrid Search

In [ ]:
resp = conn.ai.answerQuestion(query,
                        method="hybrid",
                        method_parameters = {"indices": ["DocumentChunk", "Entity"],
                                             "top_k": 5,
                                             "num_hops": 2,
                                             "num_seen_min": 3,
                                             "verbose": True})

In [ ]:
print(resp["response"])

In [ ]:
print(resp["retrieved"])

### Answer question using Similarity Search

In [ ]:
resp = conn.ai.answerQuestion(query,
                        method="similarity",
                        method_parameters={"index": "DocumentChunk",
                                           "top_k": 5,
                                           "withHyDE": False})

In [ ]:
print(resp["response"])

### Answer question using Sibling Search

In [ ]:
resp = conn.ai.answerQuestion(query,
                        method="contextual",
                        method_parameters={"index": "DocumentChunk",
                                           "top_k": 5,
                                           "lookahead": 3,
                                           "lookback": 3,
                                           "withHyDE": False})

In [ ]:
print(resp["response"])